<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/03-oop-for-ml/03_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 3: Solutions

Try the exercises first. The notes after each solution focus on the design decisions: what state the class keeps, why it's stored that way, and which patterns from real libraries it follows.

In [1]:
# Setup: run this cell once before you start.
# It defines the check functions, plus a few helper classes that some
# exercises need (Metric, and the small transformers used to test Pipeline).

import copy
import math
from abc import ABC, abstractmethod


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, float) or isinstance(b, float):
        if not isinstance(a, (int, float)) or not isinstance(b, (int, float)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=70):
    text = repr(x)
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- helpers used by the exercises ----

class Metric(ABC):
    """Base class for metrics that accumulate over batches (used in exercise 9)."""

    def __init__(self):
        self.reset()

    @abstractmethod
    def reset(self):
        """Clear all accumulated state."""

    @abstractmethod
    def update(self, y_true, y_pred):
        """Add one batch of labels and predictions."""

    @abstractmethod
    def compute(self):
        """Return the metric over everything seen since the last reset."""


class AddConstant:
    """Transformer: adds a constant to every value (used to test Pipeline)."""
    def __init__(self, c):
        self.c = c

    def fit(self, X):
        return self

    def transform(self, X):
        return [x + self.c for x in X]


class DivideByMax:
    """Transformer: divides by the largest value seen during fit."""
    def fit(self, X):
        self.max_ = max(X)
        return self

    def transform(self, X):
        return [x / self.max_ for x in X]


class Echo:
    """Estimator: 'predicts' its input unchanged, so tests can see what reached it."""
    def fit(self, X, y):
        return self

    def predict(self, X):
        return list(X)


class MeanRegressor:
    """Estimator: always predicts the mean of the training targets."""
    def fit(self, X, y):
        self.mean_ = sum(y) / len(y)
        return self

    def predict(self, X):
        return [self.mean_ for _ in X]


# ---- checks ----

def check_RunningMean(C):
    def three_values():
        r = C()
        for v in (2, 4, 6):
            r.add(v)
        return (r.count, r.mean())

    def independent():
        r1, r2 = C(), C()
        r1.add(10)
        return (r1.mean(), r2.mean(), r2.count)

    _scenarios("RunningMean", [
        ("a new object has count 0", lambda: C().count, 0, "set self.count in __init__"),
        ("mean() with no values is 0.0", lambda: C().mean(), 0.0, None),
        ("add 2, 4, 6 then check (count, mean)", three_values, (3, 4.0), None),
        ("two objects don't share state", independent, (10.0, 0.0, 0),
         "create the running total and count in __init__, not in the class body"),
    ])


def check_LabelEncoder(C):
    def fit_returns_self():
        e = C()
        return e.fit(["b", "a"]) is e

    fitted = lambda: C().fit(["dog", "cat", "dog", "bird"])
    _scenarios("LabelEncoder", [
        ("fit() returns self", fit_returns_self, True, "end fit() with `return self`"),
        ("classes_ is the sorted unique labels", lambda: fitted().classes_, ["bird", "cat", "dog"], None),
        ("transform", lambda: fitted().transform(["cat", "dog", "cat"]), [1, 2, 1], None),
        ("inverse_transform", lambda: fitted().inverse_transform([0, 2]), ["bird", "dog"], None),
        ("unknown label", lambda: fitted().transform(["fish"]), Raises(ValueError),
         "raise ValueError for labels not seen during fit"),
    ])


def check_ExponentialMovingAverage(C):
    def sequence():
        e = C(0.5)
        return [e(10), e(20), e(20)]

    def value_after():
        e = C(0.5)
        e(4)
        e(8)
        return e.value

    _scenarios("ExponentialMovingAverage", [
        ("value is None before the first call", lambda: C(0.3).value, None, None),
        ("the first call returns the value itself", lambda: C(0.1)(5.0), 5.0,
         "there's no history yet, so the average is just the first value"),
        ("alpha=0.5 over 10, 20, 20", sequence, [10.0, 15.0, 17.5], "new = alpha * x + (1 - alpha) * old"),
        ("value attribute after two calls", value_after, 6.0, None),
        ("alpha=0 is invalid", lambda: C(0), Raises(ValueError), "alpha must be in (0, 1]"),
        ("alpha=1.5 is invalid", lambda: C(1.5), Raises(ValueError), None),
    ])


def check_Subset(C):
    base = ["a", "b", "c", "d", "e"]
    s = lambda: C(base, [4, 0, 2])
    _scenarios("Subset", [
        ("len", lambda: len(s()), 3, "implement __len__"),
        ("first item", lambda: s()[0], "e", "implement __getitem__: map position i to base[indices[i]]"),
        ("last item", lambda: s()[2], "c", None),
        ("negative index", lambda: s()[-1], "c", None),
        ("iterating with a for loop", lambda: [x for x in s()], ["e", "a", "c"],
         "a for loop calls s[0], s[1], ... until IndexError. Make sure an out-of-range index raises IndexError"),
    ])


def check_MinMaxScaler(C):
    def fit_returns_self():
        sc = C()
        return sc.fit([[1, 2], [3, 4]]) is sc

    def input_untouched():
        X = [[1, 2], [3, 4]]
        C().fit_transform(X)
        return X

    _scenarios("MinMaxScaler", [
        ("fit() returns self", fit_returns_self, True, None),
        ("fit_transform", lambda: C().fit_transform([[1, 10], [3, 30], [5, 50]]),
         [[0.0, 0.0], [0.5, 0.5], [1.0, 1.0]], "scale each column separately"),
        ("transform uses the min/max learned in fit",
         lambda: C().fit([[0, 100], [10, 200]]).transform([[5, 150], [20, 0]]),
         [[0.5, 0.5], [2.0, -1.0]], "don't refit or clip on new data: values outside the training range go outside 0..1"),
        ("constant column becomes 0.0", lambda: C().fit_transform([[1, 7], [3, 7]]),
         [[0.0, 0.0], [1.0, 0.0]], None),
        ("transform before fit", lambda: C().transform([[1]]), Raises(RuntimeError), None),
        ("input is not modified", input_untouched, [[1, 2], [3, 4]], "build new lists instead of changing X"),
    ])


def check_EarlyStopping(C):
    def basic():
        es = C(patience=2)
        return [es.step(loss) for loss in [1.0, 0.8, 0.85, 0.9, 0.7]]

    def with_delta():
        es = C(patience=1, min_delta=0.1)
        return [es.step(loss) for loss in [1.0, 0.95, 0.85]]

    def tracks_best():
        es = C()
        for loss in [0.5, 0.4, 0.6]:
            es.step(loss)
        return es.best

    def nan_loss():
        es = C(patience=1)
        return [es.step(1.0), es.step(float("nan")), es.best]

    _scenarios("EarlyStopping", [
        ("best starts at infinity", lambda: C().best, float("inf"), None),
        ("patience=2", basic, [False, False, False, True, False],
         "stop after `patience` steps in a row without improvement; an improvement resets the count"),
        ("min_delta=0.1", with_delta, [False, True, False], "an improvement means loss < best - min_delta"),
        ("best is the lowest loss so far", tracks_best, 0.4, None),
        ("nan is never an improvement", nan_loss, [False, True, 1.0],
         "any comparison with nan is False. Write the check so that False means 'no improvement'"),
    ])


def check_Vector(C):
    def copies_input():
        data = [1, 2]
        v = C(data)
        data.append(3)
        return len(v)

    def unchanged():
        a = C([1, 2])
        a * 3
        a + C([1, 1])
        return a.values

    _scenarios("Vector", [
        ("repr", lambda: repr(C([1, 2, 3])), "Vector([1, 2, 3])", None),
        ("len", lambda: len(C([1, 2, 3])), 3, None),
        ("indexing", lambda: C([7, 8, 9])[1], 8, None),
        ("addition", lambda: (C([1, 2]) + C([3, 4])).values, [4, 6], None),
        ("subtraction", lambda: (C([5, 5]) - C([1, 2])).values, [4, 3], None),
        ("vector * number", lambda: (C([1, 2]) * 3).values, [3, 6], None),
        ("number * vector", lambda: (3 * C([1, 2])).values, [3, 6], "implement __rmul__"),
        ("dot product", lambda: C([1, 2, 3]).dot(C([4, 5, 6])), 32, None),
        ("equal vectors", lambda: C([1, 2]) == C([1, 2]), True, None),
        ("different vectors", lambda: C([1, 2]) == C([2, 1]), False, None),
        ("comparing with a non-Vector", lambda: C([1, 2]) == [1, 2], False, "return NotImplemented for other types"),
        ("adding vectors of different lengths", lambda: C([1, 2]) + C([1, 2, 3]), Raises(ValueError), None),
        ("operations don't modify the original", unchanged, [1, 2], "return a new Vector"),
        ("the constructor copies the list", copies_input, 2, "store list(values), not the caller's list itself"),
    ])


def check_TrainConfig(C):
    def defaults():
        c = C()
        return (c.lr, c.batch_size, c.epochs, c.hidden_sizes, c.dropout)

    def separate_lists():
        a, b = C(), C()
        a.hidden_sizes.append(16)
        return b.hidden_sizes

    _scenarios("TrainConfig", [
        ("default values", defaults, (0.001, 32, 10, [64, 32], 0.0), None),
        ("keyword arguments", lambda: C(lr=0.01, epochs=3).epochs, 3, None),
        ("instances don't share hidden_sizes", separate_lists, [64, 32], "use field(default_factory=...)"),
        ("equal configs compare equal", lambda: C(lr=0.01) == C(lr=0.01), True, "use @dataclass"),
        ("lr=0 is invalid", lambda: C(lr=0), Raises(ValueError), "validate in __post_init__"),
        ("batch_size=0 is invalid", lambda: C(batch_size=0), Raises(ValueError), None),
        ("dropout=1.0 is invalid", lambda: C(dropout=1.0), Raises(ValueError), "dropout must be in [0, 1)"),
        ("num_steps", lambda: C(batch_size=32, epochs=2).num_steps(100), 8, "100 examples in batches of 32 is 4 batches (round up)"),
    ])


def check_Accuracy(C):
    def one_batch():
        m = C()
        m.update([1, 0, 1], [1, 1, 1])
        return m.compute()

    def two_batches():
        m = C()
        m.update([1, 1], [1, 1])
        m.update([0, 0, 0, 0], [1, 1, 1, 1])
        return m.compute()

    def after_reset():
        m = C()
        m.update([1], [1])
        m.reset()
        return m.compute()

    _scenarios("Accuracy", [
        ("is a Metric", lambda: isinstance(C(), Metric), True, "class Accuracy(Metric):"),
        ("one batch", one_batch, 2 / 3, None),
        ("two batches of different sizes", two_batches, 2 / 6,
         "count correct predictions over all examples, don't average the batch accuracies"),
        ("compute() with no data", lambda: C().compute(), 0.0, None),
        ("reset() clears everything", after_reset, 0.0, None),
        ("mismatched lengths", lambda: C().update([1, 0], [1]), Raises(ValueError), None),
    ])


def check_Pipeline(C):
    def build():
        return C([("add", AddConstant(1)), ("div", DivideByMax()), ("out", Echo())])

    def fit_returns_self():
        p = build()
        return p.fit([1, 3], [0, 0]) is p

    _scenarios("Pipeline", [
        ("fit() returns self", fit_returns_self, True, None),
        ("each step is fitted on the previous step's output", lambda: build().fit([1, 3], [0, 0])["div"].max_, 4,
         "DivideByMax should see [2, 4] (after adding 1), so its max_ is 4"),
        ("predict applies the steps in order", lambda: build().fit([1, 3], [0, 0]).predict([3, 7]), [1.0, 2.0],
         "transform X with each step in order, then call predict on the last step"),
        ("access a step by name", lambda: isinstance(build()["add"], AddConstant), True, "implement __getitem__"),
        ("len", lambda: len(build()), 3, None),
        ("the final step receives y",
         lambda: C([("add", AddConstant(0)), ("m", MeanRegressor())]).fit([1, 2], [10, 20]).predict([5, 5, 5]),
         [15.0, 15.0, 15.0], None),
        ("duplicate step names", lambda: C([("a", AddConstant(1)), ("a", Echo())]), Raises(ValueError), None),
    ])


def check_WarmupCosineSchedule(C):
    s = lambda: C(1.0, warmup_steps=4, total_steps=14)
    _scenarios("WarmupCosineSchedule", [
        ("step 0 (warmup)", lambda: s()(0), 0.25, "during warmup: base_lr * (step + 1) / warmup_steps"),
        ("last warmup step", lambda: s()(3), 1.0, None),
        ("first step after warmup", lambda: s()(4), 1.0, None),
        ("halfway through the decay", lambda: s()(9), 0.5, None),
        ("final step", lambda: s()(14), 0.0, None),
        ("past the end stays at min_lr", lambda: s()(100), 0.0, "clamp progress to at most 1.0"),
        ("min_lr, halfway", lambda: C(1.0, warmup_steps=0, total_steps=10, min_lr=0.1)(5), 0.55, None),
        ("min_lr, at the end", lambda: C(1.0, warmup_steps=0, total_steps=10, min_lr=0.1)(10), 0.1, None),
        ("warmup_steps >= total_steps", lambda: C(1.0, warmup_steps=10, total_steps=10), Raises(ValueError), None),
    ])


def check_ConfusionMatrix(C):
    labels = ["cat", "dog", "bird"]
    y_true = ["cat", "cat", "dog", "bird", "dog", "cat"]
    y_pred = ["cat", "dog", "dog", "bird", "cat", "cat"]

    def filled():
        cm = C(labels)
        cm.update(y_true[:3], y_pred[:3])
        cm.update(y_true[3:], y_pred[3:])
        return cm

    def safe_matrix():
        cm = filled()
        m = cm.matrix
        m[0][0] = 99
        return cm[("cat", "cat")]

    def zero_division():
        cm = C(labels)
        cm.update(["cat"], ["cat"])
        return (cm.precision("dog"), cm.recall("dog"))

    _scenarios("ConfusionMatrix", [
        ("matrix (rows are true labels, columns are predictions)", lambda: filled().matrix,
         [[2, 1, 0], [1, 1, 0], [0, 0, 1]], "counts should add up over several update() calls. If a whole column changes at once, check how you created the rows"),
        ("cm[(true, pred)]", lambda: filled()[("cat", "dog")], 1, None),
        ("accuracy", lambda: filled().accuracy(), 4 / 6, None),
        ("precision('cat')", lambda: filled().precision("cat"), 2 / 3, "of everything predicted 'cat', how much really was 'cat'"),
        ("recall('dog')", lambda: filled().recall("dog"), 1 / 2, "of everything that really was 'dog', how much was predicted 'dog'"),
        ("precision('bird')", lambda: filled().precision("bird"), 1.0, None),
        ("no predictions or examples of a class", zero_division, (0.0, 0.0), "return 0.0 instead of dividing by zero"),
        ("changing .matrix doesn't change the counts", safe_matrix, 2, "return a copy of the rows from the matrix property"),
        ("unknown label", lambda: C(labels).update(["fish"], ["cat"]), Raises(ValueError), None),
    ])


print("Setup done. Checks are ready.")

Setup done. Checks are ready.


## Easy

### Exercise 1: `RunningMean`

In [2]:
class RunningMean:
    def __init__(self):
        self.total = 0.0
        self.count = 0

    def add(self, x):
        self.total += x
        self.count += 1

    def mean(self):
        if self.count == 0:
            return 0.0
        return self.total / self.count

In [3]:
check_RunningMean(RunningMean)

PASS  all 4 tests passed for RunningMean


The state is two numbers, both created in `__init__` so each instance gets its own. If you put `total = 0.0` in the class body instead, `self.total += x` would actually still work, because it reads the class attribute and then creates an instance attribute. That's confusing enough that it's best avoided entirely. For mutable state like a list, putting it in the class body would be a real bug, as section 3 of the lesson showed.

Using constant memory no matter how many values come in is the important design choice here. You'll see this "running statistic" idea again in batch normalization, which keeps a running mean of activations during training.

### Exercise 2: `LabelEncoder`

In [4]:
class LabelEncoder:
    def fit(self, labels):
        self.classes_ = sorted(set(labels))
        self._index = {label: i for i, label in enumerate(self.classes_)}
        return self

    def transform(self, labels):
        unknown = [label for label in labels if label not in self._index]
        if unknown:
            raise ValueError(f"unknown labels: {sorted(set(unknown))}")
        return [self._index[label] for label in labels]

    def inverse_transform(self, ids):
        return [self.classes_[i] for i in ids]

In [5]:
check_LabelEncoder(LabelEncoder)

PASS  all 5 tests passed for LabelEncoder


The class keeps two lookups, one in each direction, just like the vocabulary in Module 2: a list for id to label, and a dict for label to id. You could write `self.classes_.index(label)` in `transform` instead, but that searches the list every time, which is O(n) per label.

`_index` has a leading underscore because it's an internal detail. Users of the class should only need `classes_`.

This class doesn't have an `__init__`. That's fine: there's nothing to set up before `fit`. scikit-learn's own `LabelEncoder` works the same way, and its attributes only appear after fitting.

### Exercise 3: `ExponentialMovingAverage`

In [6]:
class ExponentialMovingAverage:
    def __init__(self, alpha):
        if not 0 < alpha <= 1:
            raise ValueError(f"alpha must be in (0, 1], got {alpha}")
        self.alpha = alpha
        self.value = None

    def __call__(self, x):
        if self.value is None:
            self.value = float(x)
        else:
            self.value = self.alpha * x + (1 - self.alpha) * self.value
        return self.value

In [7]:
check_ExponentialMovingAverage(ExponentialMovingAverage)

PASS  all 6 tests passed for ExponentialMovingAverage


`__call__` makes the object usable like a function, but unlike a function, it remembers the previous value between calls. This "function with memory" pattern is common: PyTorch modules, learning-rate schedules and metrics all use it.

`if self.value is None` rather than `if not self.value`: an average of exactly 0.0 is a legitimate value, and treating it as "no value yet" would restart the average. It's the same `is None` lesson from Module 1.

The validation happens in `__init__`, so a bad `alpha` fails as soon as the object is created, not later when it's used.

### Exercise 4: `Subset`

In [8]:
class Subset:
    def __init__(self, dataset, indices):
        self.dataset = dataset
        self.indices = list(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        return self.dataset[self.indices[i]]

In [9]:
check_Subset(Subset)

PASS  all 5 tests passed for Subset


Three short methods, and the subset now works with `len`, indexing, negative indexes and `for` loops, because each of those is built on `__len__` and `__getitem__`.

Iteration works because when `i` runs past the end, `self.indices[i]` raises an `IndexError`, and that's exactly the signal a `for` loop uses to stop. We get the right behaviour for free by delegating to a list.

The subset stores the indices, not the examples. If the underlying dataset loads images from disk, the subset doesn't load anything until you actually ask for an item. This is the composition idea from the lesson: a `Subset` *has* a dataset and adds a thin layer on top.

With this and a shuffled list of indices, splitting a dataset into train and validation is two lines:

```python
idx = list(range(len(data)))
random.shuffle(idx)
train, val = Subset(data, idx[:800]), Subset(data, idx[800:])
```

## Medium

### Exercise 5: `MinMaxScaler`

In [10]:
class MinMaxScaler:
    def __init__(self):
        self.min_ = None
        self.max_ = None

    def fit(self, X):
        columns = list(zip(*X))
        self.min_ = [min(col) for col in columns]
        self.max_ = [max(col) for col in columns]
        return self

    def transform(self, X):
        if self.min_ is None:
            raise RuntimeError("call fit() before transform()")
        return [
            [self._scale(x, lo, hi) for x, lo, hi in zip(row, self.min_, self.max_)]
            for row in X
        ]

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    @staticmethod
    def _scale(x, lo, hi):
        if hi == lo:
            return 0.0
        return (x - lo) / (hi - lo)

In [11]:
check_MinMaxScaler(MinMaxScaler)

PASS  all 6 tests passed for MinMaxScaler


`zip(*X)` turns rows into columns, the transpose trick from Module 1, so the per-column minimums and maximums are one comprehension each.

`fit_transform` is a single line because `fit` returns `self`: `self.fit(X).transform(X)`. That's the payoff of the `return self` convention.

`_scale` is a static method because it doesn't need anything from the instance. It's private (underscore) because it's a helper, not part of the public interface. Pulling it out keeps the comprehension in `transform` readable.

Compare this to the Module 1 mini project, where `mins` and `maxs` had to be passed around by hand. Now they live inside the object that uses them, and there's no way to scale the test set with the wrong numbers.

### Exercise 6: `EarlyStopping`

In [12]:
class EarlyStopping:
    def __init__(self, patience=3, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best = float("inf")
        self.bad_epochs = 0

    def step(self, loss):
        if loss < self.best - self.min_delta:
            self.best = loss
            self.bad_epochs = 0
        else:
            self.bad_epochs += 1
        return self.bad_epochs >= self.patience

In [13]:
check_EarlyStopping(EarlyStopping)

PASS  all 5 tests passed for EarlyStopping


This is the early stopping loop from the Module 1 lesson, turned into a reusable object. The training loop no longer has to manage `best` and `bad_epochs` itself. It just asks `if stopper.step(val_loss): break`.

The `nan` case is decided by which way round the `if` is written. Here, the improvement branch only runs when `loss < best - min_delta` is `True`, and any comparison with `nan` is `False`, so `nan` falls into the "no improvement" branch. That's what we want.

If you'd written it the other way round:

```python
if loss >= self.best - self.min_delta:
    self.bad_epochs += 1
else:
    self.best = loss          # nan ends up here
```

then `nan >= ...` is also `False`, so `nan` would be treated as an improvement and become the new `best`. After that, every comparison against `best` is `False` forever, and early stopping is broken. Same logic, different results, all because of `nan`. When in doubt, write the condition so that the safe behaviour is in the `else`.

### Exercise 7: `Vector`

In [14]:
class Vector:
    def __init__(self, values):
        self.values = list(values)

    def __repr__(self):
        return f"Vector({self.values})"

    def __len__(self):
        return len(self.values)

    def __getitem__(self, i):
        return self.values[i]

    def _check_same_length(self, other):
        if len(self) != len(other):
            raise ValueError(f"length mismatch: {len(self)} vs {len(other)}")

    def __add__(self, other):
        self._check_same_length(other)
        return Vector([a + b for a, b in zip(self.values, other.values)])

    def __sub__(self, other):
        self._check_same_length(other)
        return Vector([a - b for a, b in zip(self.values, other.values)])

    def __mul__(self, scalar):
        return Vector([a * scalar for a in self.values])

    def __rmul__(self, scalar):
        return self * scalar

    def __eq__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        return self.values == other.values

    def dot(self, other):
        self._check_same_length(other)
        return sum(a * b for a, b in zip(self.values, other.values))

In [15]:
check_Vector(Vector)

PASS  all 14 tests passed for Vector


`list(values)` in `__init__` makes a copy. Without it, the vector and the caller share the same list, so if the caller later appends to their list, the vector silently changes too. It's the aliasing problem from Module 1 again, and it's a common source of bugs in classes that store what they're given.

`__rmul__` just calls `__mul__`, because multiplying by a number is the same in either order. `__eq__` returns `NotImplemented` for non-vectors. Python then falls back to its default comparison, which gives `False`.

The length check is pulled out into `_check_same_length` so that `+`, `-` and `dot` share one implementation and one error message.

Note what's missing: we didn't define `__hash__`. When a class defines `__eq__` but not `__hash__`, Python makes its instances unhashable, so they can't go in a set or be dict keys. That's actually right for a mutable object like this one, for the reasons from Module 2.

NumPy arrays are essentially this class written in C and generalised to any number of dimensions, which makes them far faster. Module 6 starts there.

### Exercise 8: `TrainConfig`

In [16]:
from dataclasses import dataclass, field


@dataclass
class TrainConfig:
    lr: float = 1e-3
    batch_size: int = 32
    epochs: int = 10
    hidden_sizes: list = field(default_factory=lambda: [64, 32])
    dropout: float = 0.0

    def __post_init__(self):
        if self.lr <= 0:
            raise ValueError(f"lr must be positive, got {self.lr}")
        if self.batch_size < 1:
            raise ValueError(f"batch_size must be at least 1, got {self.batch_size}")
        if not 0 <= self.dropout < 1:
            raise ValueError(f"dropout must be in [0, 1), got {self.dropout}")

    def num_steps(self, n_examples):
        batches_per_epoch = (n_examples + self.batch_size - 1) // self.batch_size
        return batches_per_epoch * self.epochs

In [17]:
check_TrainConfig(TrainConfig)

PASS  all 8 tests passed for TrainConfig


The `@dataclass` decorator generated `__init__`, `__repr__` and `__eq__` from the field list, and we only wrote the parts that are specific to this config: validation and one derived value.

`field(default_factory=lambda: [64, 32])` calls the lambda each time a new instance is created, so every config gets a fresh list. A plain `= [64, 32]` would be shared between all instances, which is why dataclasses refuse it with an error.

`num_steps` is the kind of number you need for learning-rate schedules, which usually need to know the total step count up front. You'll use exactly this with the schedule in exercise 11.

`(n + b - 1) // b` is integer division rounded up, the same trick as `DataLoader.__len__` in the lesson. `math.ceil(n / b)` also works, but goes through a float, which can be off for very large numbers.

### Exercise 9: `Accuracy`

In [18]:
class Accuracy(Metric):
    def reset(self):
        self.correct = 0
        self.total = 0

    def update(self, y_true, y_pred):
        if len(y_true) != len(y_pred):
            raise ValueError(f"length mismatch: {len(y_true)} vs {len(y_pred)}")
        self.correct += sum(t == p for t, p in zip(y_true, y_pred))
        self.total += len(y_true)

    def compute(self):
        if self.total == 0:
            return 0.0
        return self.correct / self.total

In [19]:
check_Accuracy(Accuracy)

PASS  all 6 tests passed for Accuracy


We didn't write `__init__`. The base class's `__init__` calls `self.reset()`, and since `self` is an `Accuracy`, that runs *our* `reset`, which sets up `correct` and `total`. The parent defines when things happen, and the subclass fills in what happens. This is called the *template method* pattern, and frameworks use it constantly.

Storing counts instead of per-batch accuracies is what makes the result correct. With a batch of 2 that's 100% right and a batch of 4 that's 0% right, the true accuracy is 2 out of 6, or 33%. Averaging the batch accuracies would give 50%. The last batch of an epoch is often smaller than the others, so this mistake does happen in real training code.

Because every metric shares the same interface, a training loop can handle a whole list of them without knowing which ones they are:

```python
for metric in metrics:
    metric.update(yb, preds)
```

## Hard

### Exercise 10: `Pipeline`

In [20]:
class Pipeline:
    def __init__(self, steps):
        names = [name for name, _ in steps]
        if len(set(names)) != len(names):
            raise ValueError(f"step names must be unique, got {names}")
        self.steps = list(steps)
        self._by_name = dict(steps)

    def __len__(self):
        return len(self.steps)

    def __getitem__(self, name):
        return self._by_name[name]

    def fit(self, X, y):
        *transformers, (_, estimator) = self.steps
        for _, step in transformers:
            X = step.fit(X).transform(X)
        estimator.fit(X, y)
        return self

    def predict(self, X):
        *transformers, (_, estimator) = self.steps
        for _, step in transformers:
            X = step.transform(X)
        return estimator.predict(X)

In [21]:
check_Pipeline(Pipeline)

PASS  all 7 tests passed for Pipeline


`*transformers, (_, estimator) = self.steps` uses star unpacking from Module 2 to split the list into "everything but the last" and "the last one", and unpacks the last pair at the same time.

The key line is `X = step.fit(X).transform(X)`. Each transformer is fitted on the output of the previous one, which is what the "each step is fitted on the previous step's output" check tests. If you fit every step on the original `X`, `DivideByMax` learns the wrong maximum.

`predict` only calls `transform`, never `fit`. That's the whole point of a pipeline: all the learned numbers come from the training data, and the same steps are replayed exactly on new data. It makes the data leakage mistake from Module 1 (scaling the test set with its own statistics) impossible.

Notice that `Pipeline` doesn't know or care what its steps are. It works with any objects that have the right methods. Python calls this *duck typing*: if it has `fit` and `transform`, it's a transformer. There's no need for a shared base class, and that's why scikit-learn pipelines work with third-party models too.

### Exercise 11: `WarmupCosineSchedule`

In [22]:
import math


class WarmupCosineSchedule:
    def __init__(self, base_lr, warmup_steps, total_steps, min_lr=0.0):
        if not 0 <= warmup_steps < total_steps:
            raise ValueError("need 0 <= warmup_steps < total_steps")
        self.base_lr = base_lr
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.min_lr = min_lr

    def __call__(self, step):
        if step < self.warmup_steps:
            return self.base_lr * (step + 1) / self.warmup_steps
        progress = (step - self.warmup_steps) / (self.total_steps - self.warmup_steps)
        progress = min(progress, 1.0)
        cosine = 0.5 * (1 + math.cos(math.pi * progress))
        return self.min_lr + (self.base_lr - self.min_lr) * cosine

    def __repr__(self):
        return (f"WarmupCosineSchedule(base_lr={self.base_lr}, warmup_steps={self.warmup_steps}, "
                f"total_steps={self.total_steps}, min_lr={self.min_lr})")

In [23]:
check_WarmupCosineSchedule(WarmupCosineSchedule)

PASS  all 9 tests passed for WarmupCosineSchedule


All the validation happens once in `__init__`, which also guarantees `total_steps - warmup_steps` is never zero, so `__call__` can't divide by zero. When `warmup_steps` is 0, the `step < self.warmup_steps` branch never runs, so that division is safe too.

`cosine` goes from 1 at the start of the decay to 0 at the end, following half a cosine wave. Multiplying it into the range between `min_lr` and `base_lr` gives a smooth curve from `base_lr` down to `min_lr`.

Why warm up at all? At the very start of training, the weights are random and the gradients can be large and unreliable. A big learning rate right away can push the model somewhere it never recovers from. Starting small and ramping up avoids that. The slow cosine decay at the end lets the model settle into a good minimum instead of bouncing around it.

Here's what the whole schedule looks like, drawn with text so we don't need a plotting library yet (that's Module 8):

In [24]:
schedule = WarmupCosineSchedule(1.0, warmup_steps=5, total_steps=40, min_lr=0.05)
for step in range(0, 41, 2):
    lr = schedule(step)
    print(f"step {step:2d}  {lr:.3f}  " + "#" * round(lr * 40))

step  0  0.200  ########
step  2  0.600  ########################
step  4  1.000  ########################################
step  6  0.998  ########################################
step  8  0.983  #######################################
step 10  0.953  ######################################
step 12  0.909  ####################################
step 14  0.853  ##################################
step 16  0.787  ###############################
step 18  0.712  ############################
step 20  0.631  #########################
step 22  0.546  ######################
step 24  0.461  ##################
step 26  0.378  ###############
step 28  0.300  ############
step 30  0.229  #########
step 32  0.167  #######
step 34  0.117  #####
step 36  0.080  ###
step 38  0.058  ##
step 40  0.050  ##


PyTorch and Hugging Face both provide schedules like this one. Having written it yourself, you know exactly what the `warmup_steps` and `num_training_steps` arguments to those functions mean, and why `num_training_steps` needs to be right: if it's too small, the learning rate hits its minimum halfway through training and stays there.

### Exercise 12: `ConfusionMatrix`

In [25]:
class ConfusionMatrix:
    def __init__(self, labels):
        self.labels = list(labels)
        self._index = {label: i for i, label in enumerate(self.labels)}
        n = len(self.labels)
        self._counts = [[0] * n for _ in range(n)]

    def update(self, y_true, y_pred):
        unknown = (set(y_true) | set(y_pred)) - set(self._index)
        if unknown:
            raise ValueError(f"unknown labels: {sorted(unknown, key=str)}")
        for t, p in zip(y_true, y_pred):
            self._counts[self._index[t]][self._index[p]] += 1

    @property
    def matrix(self):
        return [row.copy() for row in self._counts]

    def __getitem__(self, key):
        true_label, pred_label = key
        return self._counts[self._index[true_label]][self._index[pred_label]]

    def accuracy(self):
        total = sum(sum(row) for row in self._counts)
        correct = sum(self._counts[i][i] for i in range(len(self.labels)))
        return correct / total if total else 0.0

    def precision(self, label):
        j = self._index[label]
        predicted = sum(row[j] for row in self._counts)        # column sum
        return self._counts[j][j] / predicted if predicted else 0.0

    def recall(self, label):
        i = self._index[label]
        actual = sum(self._counts[i])                         # row sum
        return self._counts[i][i] / actual if actual else 0.0

    def __repr__(self):
        width = max(len(str(label)) for label in self.labels) + 2
        header = " " * width + "".join(f"{str(l):>{width}}" for l in self.labels)
        rows = [f"{str(l):>{width}}" + "".join(f"{c:>{width}}" for c in row)
                for l, row in zip(self.labels, self._counts)]
        return "\n".join([header] + rows)

In [26]:
check_ConfusionMatrix(ConfusionMatrix)

PASS  all 9 tests passed for ConfusionMatrix


A few design decisions worth looking at:

- **The counts are private** (`_counts`), and the public `matrix` property returns a copy of each row. If it returned `self._counts` directly, a caller could change the stored counts by accident, which is the check that tests this. This is a common use of properties: give read access without giving write access.
- **`[[0] * n for _ in range(n)]`**, not `[[0] * n] * n`. That's the list-of-lists trap from Module 2. With the second version, every row would be the same list.
- **Label validation uses set operations**: the union of everything in `y_true` and `y_pred`, minus the known labels. One line, and it reports every unknown label at once.
- **Precision is a column sum, recall is a row sum.** Rows are true labels, columns are predictions. Precision asks "of everything in this column (predicted as this label), how much is on the diagonal?" Recall asks the same about the row.

Here's the `__repr__` in action, which makes the matrix much easier to read than a list of lists:

In [27]:
cm = ConfusionMatrix(["cat", "dog", "bird"])
cm.update(["cat", "cat", "dog", "bird", "dog", "cat"],
          ["cat", "dog", "dog", "bird", "cat", "cat"])
print(cm)
print()
for label in cm.labels:
    print(f"{label:5s} precision={cm.precision(label):.2f}  recall={cm.recall(label):.2f}")

         cat   dog  bird
   cat     2     1     0
   dog     1     1     0
  bird     0     0     1

cat   precision=0.67  recall=0.67
dog   precision=0.50  recall=0.50
bird  precision=1.00  recall=1.00


Reading it: the "cat" row says that of 3 real cats, 2 were predicted as cat and 1 as dog. The off-diagonal cells show you exactly which classes the model confuses with which, which is often more useful than any single number. If a model keeps mixing up two particular classes, that tells you where to look: maybe those classes need more training data, or maybe they're genuinely hard to tell apart.

## What's next

The mini project in `04_mini_project.ipynb` builds a tiny version of the engine behind PyTorch's `loss.backward()`, using the dunder methods from this module, and uses it to train a neural network from scratch.